# DAY 04 - TASK 02: KHẢO SÁT BIẾN THỂ KIẾN TRÚC CNN (ABLATION STUDY) TRÊN FASHION-MNIST

### Mục tiêu thực hành:
1. **Thực hiện nghiên cứu loại trừ (Ablation Study)**: Khảo sát ảnh hưởng của từng thành phần kiến trúc: kích thước kernel ($3 \times 3$ vs $5 \times 5$), độ rộng số kênh (8 vs 16 vs 32), Batch Normalization và Dropout.
2. **Lựa chọn mô hình tối ưu dựa trên Validation**: Chọn mô hình có Validation Accuracy cao nhất (kết hợp Validation Loss thấp nhất).
3. **Đánh giá khách quan trên Official Test Set (10,000 ảnh)**: Kiểm chứng độ chính xác thực tế, ma trận nhầm lẫn và trực quan hóa 5 ảnh đoán đúng / 5 ảnh đoán sai.



## Bước 1: Thiết lập môi trường và cấu hình Thí nghiệm
Import các module từ thư viện lõi `srcs.libraries` và pipeline `srcs.pipelines.day04_task02_cnn_variants`.



In [ ]:
import sys
import os
import torch
import pandas as pd
import numpy as np
from IPython.display import Image, display

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from srcs.libraries.seeding import seed_everything
from srcs.libraries.training import device
from srcs.pipelines.day04_task02_cnn_variants import (
    get_ablation_configurations,
    prepare_datasets,
    create_loaders,
    run_ablation_study,
    render_comparative_learning_curves,
    render_test_confusion_matrix,
    render_prediction_samples,
    CLASS_NAMES
)

seed_everything(42)
comp_device = torch.device(device())
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
print(f"PyTorch Version: {torch.__version__} | Device: {comp_device}")
print(f"Thư mục lưu trữ dữ liệu (Gốc dự án): {DATA_DIR}")



## Bước 2: Định nghĩa 6 cấu hình Biến thể CNN (Ablation Study)
Mỗi cấu hình chỉ thay đổi đúng một thuộc tính so với `baseline`:
1. `baseline`: channels = 16, kernel_size = 3, batchnorm = False, dropout = 0.0
2. `kernel5`: kernel_size = 5 (tăng trường tiếp nhận receptive field)
3. `ch8`: channels = 8 (giảm một nửa dung lượng tham số)
4. `ch32`: channels = 32 (gấp đôi số kênh bộ lọc)
5. `batchnorm`: bổ sung Batch Normalization 2D sau mỗi tầng Conv
6. `dropout0.3`: bổ sung Dropout (p = 0.3) trước tầng Linear cuối



In [ ]:
configs = get_ablation_configurations()
df_configs = pd.DataFrame(configs)
display(df_configs)



## Bước 3: Huấn luyện và Đánh giá 6 cấu hình trên cùng điều kiện
Huấn luyện 5 epochs với cùng tập dữ liệu lưu trữ tại thư mục `data/` ở gốc dự án (6,000 train, 1,200 val, 10,000 test), seed 42 và optimizer Adam (lr = 1e-3).



In [ ]:
study_data = run_ablation_study(epochs=5, data_dir=DATA_DIR, seed=42)
results = study_data["results"]
best_res = study_data["best_result"]
test_loader = study_data["test_loader"]



## Bước 4: Bảng so sánh tổng hợp 6 biến thể CNN


In [ ]:
df_summary = pd.DataFrame([
    {
        "Cấu hình": res["config"]["name"],
        "Số tham số": f"{res['params']:,}",
        "Train Loss": f"{res['final_train_loss']:.4f}",
        "Val Loss": f"{res['final_val_loss']:.4f}",
        "Val Accuracy": f"{res['final_val_acc'] * 100:.2f}%",
        "Giây/epoch": f"{res['sec_per_epoch']:.2f}s"
    }
    for res in results
])
display(df_summary)



## Bước 5: Trực quan hóa đối chiếu Đường cong học tập (3 Subplots)
Lưu biểu đồ vào `docs/day04/task02/image_learning_curves.png` và `bin/day04/task02/results/learning_curves.png`.



In [ ]:
curves_paths = [
    os.path.join(PROJECT_ROOT, "docs", "day04", "task02", "image_learning_curves.png"),
    os.path.join(PROJECT_ROOT, "bin", "day04", "task02", "results", "learning_curves.png")
]
render_comparative_learning_curves(results, epochs=5, output_paths=curves_paths)
display(Image(curves_paths[0]))



## Bước 6: Lựa chọn Mô hình tối ưu trên tập Validation
Tiêu chí lựa chọn: Validation Accuracy cao nhất (nếu bằng nhau thì chọn Validation Loss thấp hơn).



In [ ]:
print(f"Mô hình được chọn: {best_res['config']['name'].upper()}")
print(f"  - Validation Accuracy: {best_res['final_val_acc'] * 100:.2f}%")
print(f"  - Validation Loss:     {best_res['final_val_loss']:.4f}")
print(f"  - Tổng số tham số:     {best_res['params']:,}")



## Bước 7: Kiểm định Mô hình tối ưu trên Official Test Set và Ma trận nhầm lẫn
Tập Test chính thức (10,000 ảnh) chỉ được đem ra đánh giá đúng một lần duy nhất cho mô hình chiến thắng (`batchnorm`).
Lưu ảnh vào `docs/day04/task02/image_confusion_matrix_test.png` và `bin/day04/task02/results/confusion_matrix_test.png`.



In [ ]:
test_cm_paths = [
    os.path.join(PROJECT_ROOT, "docs", "day04", "task02", "image_confusion_matrix_test.png"),
    os.path.join(PROJECT_ROOT, "bin", "day04", "task02", "results", "confusion_matrix_test.png")
]
x_test, y_true, y_pred, test_acc = render_test_confusion_matrix(
    model=best_res["model"],
    test_loader=test_loader,
    comp_device=comp_device,
    model_name=best_res["config"]["name"],
    output_paths=test_cm_paths
)
print(f"Official Test Accuracy ({best_res['config']['name']}): {test_acc * 100:.2f}%")
display(Image(test_cm_paths[0]))



## Bước 8: Trực quan hóa 5 mẫu ảnh dự đoán Đúng và 5 mẫu ảnh dự đoán Sai trên Test Set
Lưu ảnh mẫu vào `docs/day04/task02/` và `bin/day04/task02/results/`.



In [ ]:
correct_indices = np.where(y_true == y_pred)[0][:5]
wrong_indices = np.where(y_true != y_pred)[0][:5]

correct_paths = [
    os.path.join(PROJECT_ROOT, "docs", "day04", "task02", "image_five_correct.png"),
    os.path.join(PROJECT_ROOT, "bin", "day04", "task02", "results", "five_correct.png")
]
render_prediction_samples(x_test, y_true, y_pred, correct_indices, "5 ảnh dự đoán ĐÚNG trên tập Test", correct_paths)
display(Image(correct_paths[0]))

wrong_paths = [
    os.path.join(PROJECT_ROOT, "docs", "day04", "task02", "image_five_wrong.png"),
    os.path.join(PROJECT_ROOT, "bin", "day04", "task02", "results", "five_wrong.png")
]
render_prediction_samples(x_test, y_true, y_pred, wrong_indices, "5 ảnh dự đoán SAI trên tập Test", wrong_paths)
display(Image(wrong_paths[0]))



## Bước 9: Kết luận và Khuyến nghị Thực hành Tốt nhất

1. **Hiệu quả vượt trội của Batch Normalization:**
   - `batchnorm` chỉ thêm 96 tham số nhưng đem lại mức nhảy vọt ấn tượng nhất: đạt **88.00% val acc** và **86.59% test acc**.
   - Việc chuẩn hóa giúp mạng hội tụ thần tốc ngay từ epoch 1 (đạt 85.67% so với 76.92% của baseline) nhờ triệt tiêu trôi dạt phân phối đầu vào.

2. **Quy luật về số lượng kênh:**
   - Giảm xuống 8 kênh (`ch8`): Mô hình bị thiếu dung lượng biểu diễn (underfit), val accuracy giảm xuống 83.75%.
   - Tăng lên 32 kênh (`ch32`): Số tham số tăng hơn gấp 2 lần (421k) và chạy chậm hơn gấp 2 lần (3.45s), nhưng accuracy (87.58%) vẫn đứng sau `batchnorm`.

3. **Receptive Field và Điều chuẩn:**
   - Kích thước kernel $5 \times 5$ (`kernel5`) không tạo khác biệt lớn so với $3 \times 3$ trên ảnh $28 \times 28$.
   - `dropout0.3` chưa phát huy hiệu quả khi số lượng epoch còn nhỏ (5 epochs), nhưng sẽ bảo vệ mô hình rất tốt khi huấn luyện dài hơi (20+ epochs).

